# Imports

In [1]:
import numpy as np
import pandas as pd
import category_encoders as ce


from tqdm import tqdm
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, log_loss
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold, cross_val_score
from sklearn.feature_selection import SequentialFeatureSelector, SelectFromModel, RFECV

from sklearn.utils.validation import check_X_y, check_array, check_is_fitted

## Utils

# Loading Dataset

In [2]:
X_train_raw = pd.read_parquet('../data/processed/X_train_raw.parquet')
X_train_fe = pd.read_parquet('../data/processed/X_train_fe.parquet')

y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test_raw = pd.read_parquet('../data/processed/X_test_raw.parquet')
X_test_fe = pd.read_parquet('../data/processed/X_test_fe.parquet')

In [3]:
X_train_fe.head()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,...,water_per_min,calories_per_step,cardio_load,cardio_efficiency,heart_step,bmi_activity,bmi_steps,bmi_calorie,sleep_stress_interaction,water_to_calorie_ratio
id,,,,,,,,,,,,,,,,,,,,,
0,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,...,0.089423,1.638282,1397.88,30.363128,0.053203,1.233654,0.019337,0.011798,average_high,0.000855
1,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,...,0.024754,0.198746,3557.87,27.192254,0.007208,0.507662,0.002612,0.013137,average_low,0.000641
2,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,...,0.040921,0.189069,2872.74,35.183246,0.005304,0.627621,0.001726,0.009126,poor_high,0.000595
3,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,...,0.033169,0.366551,4624.28,33.631714,0.010760,0.379803,0.003224,0.008791,average_high,0.000768
4,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,missing,average,...,0.047872,0.388762,3376.40,34.408602,0.011147,0.605106,0.004319,0.011105,average_missing,0.000879


In [4]:
cat_features_fe = X_train_fe.select_dtypes(include=['category', 'object']).columns.tolist()
cat_features_raw = X_train_raw.select_dtypes(include=['category', 'object']).columns.tolist()

In [5]:
cat_features_fe

['diet_type',
 'stress_level',
 'sleep_quality',
 'physical_activity_level',
 'smoking_alcohol',
 'gender',
 'sleep_stress_interaction']

In [6]:
cat_features_raw

['diet_type',
 'stress_level',
 'sleep_quality',
 'physical_activity_level',
 'smoking_alcohol',
 'gender']

In [7]:
for feat in cat_features_raw:
    X_train_raw[feat] = X_train_raw[feat].astype('category')
    X_train_raw[feat] = X_train_raw[feat].astype('category')

In [8]:
for feat in cat_features_fe:
    X_train_fe[feat] = X_train_fe[feat].astype('category')
    X_train_fe[feat] = X_train_fe[feat].astype('category')

In [13]:
print(X_train_fe.head(5).to_string())

    sleep_duration  heart_rate    bmi  calorie_expenditure  step_count  exercise_duration  water_intake diet_type stress_level sleep_quality physical_activity_level smoking_alcohol  gender  calories_per_min  steps_per_min  water_per_min  calories_per_step  cardio_load  cardio_efficiency  heart_step  bmi_activity  bmi_steps  bmi_calorie sleep_stress_interaction  water_to_calorie_ratio
id                                                                                                                                                                                                                                                                                                                                                                                                
0             5.22        70.6  25.66               2174.0      1326.0               19.8          1.86       veg         high       average               sedentary             yes  female        104.519231      63.750000     

In [15]:
print(y_train.head(5).to_string())

    health_condition
id                  
0                  2
1                  0
2                  2
3                  2
4                  0


In [9]:
y_train.head()

,health_condition
id,
0,2
1,0
2,2
3,2
4,0


# Feature Selection

## Base Model

In [10]:
lgbm = LGBMClassifier(
    objective="multiclass",
    metric="multi_logloss",
    num_class=3,
    boosting_type="gbdt",
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    feature_fraction=0.85,
    bagging_fraction=0.8,
    bagging_freq=1,
    min_child_samples=30,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    verbosity=-1,
    n_jobs=1
)

xgb = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    num_class=3,
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1,
    random_state=42,
    n_jobs=1,
    enable_categorical=True
)

cat = CatBoostClassifier(
    loss_function="MultiClass",
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3,
    random_seed=42,
    verbose=False,
    # thread_count=1,
)

models = {
    'lgbm': lgbm,
    'xgb': xgb,
     'cat': cat,
}

In [11]:
for model_str, model in tqdm(models.items()):
    
    cv_result = cross_val_score(
        model, 
        X_train_raw, 
        y_train.health_condition, 
        scoring='neg_log_loss', 
        n_jobs=1 if model_str == 'cat' else -1,
        cv=StratifiedKFold(n_splits=3, random_state=42, shuffle=True),
        params={'cat_features': cat_features_raw} if model_str == 'cat' else None 
    ).mean()

    print(f"{model_str}: {cv_result}")

 33%|█████████████████████████████████████████                                                                                  | 1/3 [04:57<09:54, 297.10s/it]

lgbm: -0.0901650457417383


 67%|██████████████████████████████████████████████████████████████████████████████████                                         | 2/3 [11:23<05:49, 349.59s/it]

xgb: -0.09033325562874477


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [35:41<00:00, 713.90s/it]

cat: -0.09047447618942177


In [12]:
for model_str, model in tqdm(models.items()):
    
    cv_result = cross_val_score(
        model, 
        X_train_fe, 
        y_train.health_condition, 
        scoring='neg_log_loss',
        n_jobs=1 if model_str == 'cat' else -1,
        cv=StratifiedKFold(n_splits=3, random_state=42, shuffle=True),
        params={'cat_features': cat_features_fe} if model_str == 'cat' else None 
    ).mean()

    print(f"{model_str}: {cv_result}")

 33%|█████████████████████████████████████████                                                                                  | 1/3 [06:38<13:16, 398.22s/it]

lgbm: -0.09090656287823816


 67%|██████████████████████████████████████████████████████████████████████████████████                                         | 2/3 [15:02<07:40, 460.50s/it]

xgb: -0.09124468018611272


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [44:00<00:00, 880.16s/it]

cat: -0.09059156130155237
